# Slide: Defect Surface Site Analysis — Detecting Sites Inside the Void

Produces three trajectory files and matching figures for one slide:

| File | Panel | Description |
|---|---|---|
| `panel1_void_ring.traj` | Left | Slab + undercoordinated ring atoms (He) + void centroid (Ne) |
| `panel2_drop_steps.traj` | Middle | Noble-gas probe descending into void, one frame per z step |
| `panel3_void_sites.traj` | Right | One frame per unique probe position inside the void |

**Edit Cell 2 only, then Run All.**

In [1]:
# Cell 1: imports — do not edit
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from ase import Atoms
from ase.io import read, write, Trajectory
from ase.neighborlist import NeighborList
from ase.geometry import get_distances

import site_analysis as sa
import importlib
importlib.reload(sa)
print('site_analysis loaded')

ModuleNotFoundError: No module named 'yaml'

In [ ]:
# Cell 2: user inputs — edit here
xyz_path      = 'pt111_defect_middle.xyz'
tag_symbol    = 'Ne'   # noble-gas probe for the drop
dz            = 0.1    # Å per drop step
stable_steps  = 50     # stop when CN does not improve for this many steps (50 = 5 Å patience)
max_drop      = 25.0   # Å maximum total drop distance (must be > depth of void + 3 Å start offset)
min_clearance = 1.3    # Å minimum probe-to-slab distance before stopping
margin        = 0.25   # Å added to NN distance to set CN cutoff radius
uniq_tol      = 0.25   # Å tolerance for unique-position deduplication

## Panel 1 — Void ring: top-view of undercoordinated atoms

**Method:**
1. Identify top-layer atoms.
2. Build in-plane neighbor graph; atoms with degree ≤ median − 1 are undercoordinated.
3. Cluster undercoordinated atoms; their centroid = void position.

**Trajectory frames:**
- Frame 0: clean slab
- Frame 1: slab + **He** at each undercoordinated atom (0.5 Å above, for visibility)
- Frame 2: Frame 1 + **Ne** at void centroid

In [ ]:
# Panel 1 — void ring trajectory
slab  = read(xyz_path)
nslab = len(slab)

# top-surface indices and in-plane NN distance
top     = sa._top_surface_indices(slab, nslab)
use_pbc = sa._has_reasonable_cell(slab)
d_nn    = sa._estimate_nn_distance_xy(slab, top, use_pbc_xy=use_pbc)
cutoff  = 1.25 * d_nn

# build neighbor list and compute in-plane degrees for top atoms
nl = NeighborList([cutoff] * nslab, self_interaction=False, bothways=True, skin=0.0)
nl.update(slab)

top_set = set(top.tolist())
deg = np.zeros(len(top), dtype=int)
for ii, a in enumerate(top):
    neigh, _ = nl.get_neighbors(a)
    deg[ii]  = sum(1 for j in neigh if j in top_set)

expected   = int(np.median(deg))
under_mask = deg <= (expected - 1)
under_top  = top[under_mask]   # slab indices of the void-ring atoms

print(f'Top-layer atoms  : {len(top)}')
print(f'Expected degree  : {expected}')
print(f'Undercoordinated : {len(under_top)}')

# detect vacancy centroid
defect_sites, _ = sa.detect_vacancy_sites_from_coordination(slab, nslab)
if not defect_sites:
    raise RuntimeError('No vacancy detected. Check xyz_path or try a different slab.')
void_pos = defect_sites[0]['position']
print(f'\nVoid centroid: x={void_pos[0]:.3f}  y={void_pos[1]:.3f}  z={void_pos[2]:.3f}')

# build frames
frame0 = slab.copy()

frame1 = slab.copy()
for idx in under_top:
    pos      = slab.positions[idx].copy()
    pos[2]  += 0.5          # small z-offset so He sits above the Pt atom
    frame1  += Atoms('He', positions=[pos])

frame2  = frame1.copy()
frame2 += Atoms(tag_symbol, positions=[void_pos])

traj1 = Trajectory('panel1_void_ring.traj', 'w')
for f in [frame0, frame1, frame2]:
    traj1.write(f)
traj1.close()

print('\nWrote: panel1_void_ring.traj')
print('  Frame 0 – clean slab')
print('  Frame 1 – slab + He at undercoordinated ring atoms')
print('  Frame 2 – Frame 1 + Ne at void centroid')

In [ ]:
# Panel 1 — top-view matplotlib figure
top_pos   = slab.positions[top]
under_pos = slab.positions[under_top]

fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(top_pos[:, 0], top_pos[:, 1],
           c='steelblue', s=220, edgecolors='k', linewidths=0.6,
           zorder=2, label='surface atoms')

ax.scatter(under_pos[:, 0], under_pos[:, 1],
           c='tomato', s=260, edgecolors='k', linewidths=1.2,
           zorder=3, label=f'undercoordinated (deg ≤ {expected-1})')

ax.scatter(void_pos[0], void_pos[1],
           c='gold', s=350, marker='*', edgecolors='k', linewidths=0.9,
           zorder=4, label='void centroid')

ax.set_xlabel('x (Å)', fontsize=12)
ax.set_ylabel('y (Å)', fontsize=12)
ax.set_title('Panel 1: Top view — void ring detection', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('panel1_void_ring.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: panel1_void_ring.png')

## Panel 2 — Drop trajectory: probe descending into the void

**Method:**
1. Place noble-gas probe 3 Å above the void centroid.
2. Lower probe by `dz` each step; count coordination number (CN) to slab atoms.
3. Stop when CN is stable for `stable_steps` steps, or probe is too close to slab.

**Trajectory:** one frame per z step (probe moving into void).  
**Figure:** CN vs z, with max-CN step highlighted.

In [ ]:
# Panel 2 — drop trajectory
NOBLE_GASES = {'He', 'Ne', 'Ar', 'Kr', 'Xe', 'Rn'}

def _estimate_first_nn(atoms):
    idx = [i for i, a in enumerate(atoms) if a.symbol not in NOBLE_GASES]
    if len(idx) < 2:
        return 3.0
    pos = atoms.positions[idx]
    _, dmat = get_distances(pos, pos, cell=atoms.cell, pbc=atoms.pbc)
    dmat = np.array(dmat)
    np.fill_diagonal(dmat, np.inf)
    nn = np.min(dmat, axis=1)
    nn = nn[np.isfinite(nn)]
    return float(np.median(nn)) if len(nn) else 3.0

def _tag_cn(atoms, i_tag, cutoff):
    # Direct distance check — avoids ASE NeighborList's pair-cutoff doubling
    # (NeighborList uses cutoff[i]+cutoff[j], which would double the threshold)
    idx = [j for j, a in enumerate(atoms) if j != i_tag and a.symbol not in NOBLE_GASES]
    if not idx:
        return 0, []
    _, d = get_distances(
        atoms.positions[i_tag:i_tag+1],
        atoms.positions[idx],
        cell=atoms.cell, pbc=atoms.pbc,
    )
    d = np.array(d).ravel()
    neigh = [idx[k] for k in range(len(idx)) if d[k] < cutoff]
    return len(neigh), neigh

def _min_dist_to_slab(atoms, i_tag):
    idx = [j for j, a in enumerate(atoms) if j != i_tag and a.symbol not in NOBLE_GASES]
    if not idx:
        return np.inf
    _, d = get_distances(atoms.positions[i_tag:i_tag+1], atoms.positions[idx],
                         cell=atoms.cell, pbc=atoms.pbc)
    return float(np.min(d))

# CN cutoff: probe is a neighbor of a slab atom if distance < d1 + margin
d1     = _estimate_first_nn(slab)
cn_cut = d1 + margin
print(f'NN distance: {d1:.3f} Å   CN cutoff: {cn_cut:.3f} Å')
print(f'(NeighborList would have used 2×{cn_cut:.3f} = {2*cn_cut:.3f} Å — too large)')

# place probe 3 Å above void centroid
z_start   = void_pos[2] + 3.0
start_pos = np.array([void_pos[0], void_pos[1], z_start])

atoms = slab.copy() + Atoms(tag_symbol, positions=[start_pos])
i_tag = len(atoms) - 1

frames_drop = []
meta_drop   = []
max_cn      = -1
best_step   = 0
no_improve  = 0

for step in range(int(max_drop / dz) + 1):
    cn, neigh = _tag_cn(atoms, i_tag, cn_cut)
    mind = _min_dist_to_slab(atoms, i_tag)
    z    = float(atoms.positions[i_tag, 2])

    frames_drop.append(atoms.copy())
    meta_drop.append({'step': step, 'z': z, 'cn': cn, 'min_dist': mind})

    if mind < min_clearance:
        print(f'  Stopped at step {step}: min_clearance reached ({mind:.3f} Å)')
        break

    if cn > max_cn:
        max_cn, best_step, no_improve = cn, step, 0
    elif cn > 0:          # only count stability once probe has made surface contact
        no_improve += 1
    if no_improve >= stable_steps:
        print(f'  Stopped at step {step}: CN stable for {stable_steps} steps')
        break

    atoms.positions[i_tag, 2] -= dz

print(f'Drop: {len(frames_drop)} frames   max CN = {max_cn} at step {best_step} (z = {meta_drop[best_step]["z"]:.3f} Å)')

write('panel2_drop_steps.traj', frames_drop)
print('Wrote: panel2_drop_steps.traj')

In [ ]:
# Panel 2 — CN vs z figure
z_vals  = [m['z']  for m in meta_drop]
cn_vals = [m['cn'] for m in meta_drop]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(z_vals, cn_vals, 'o-', color='steelblue', ms=4, lw=1.8, label='CN during drop')
ax.axvline(z_vals[best_step], color='tomato', ls='--', lw=1.5,
           label=f'max CN = {max_cn}')
ax.scatter([z_vals[best_step]], [cn_vals[best_step]],
           c='gold', s=140, zorder=5, edgecolors='k', linewidths=0.8)

# annotate start and end
ax.annotate('probe above surface', xy=(z_vals[0], cn_vals[0]),
            xytext=(z_vals[0] - 0.3, cn_vals[0] + 0.4),
            fontsize=8, arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate(f'max CN site\nz = {z_vals[best_step]:.2f} Å',
            xy=(z_vals[best_step], cn_vals[best_step]),
            xytext=(z_vals[best_step] + 0.5, cn_vals[best_step] - 0.8),
            fontsize=8, color='tomato', arrowprops=dict(arrowstyle='->', color='tomato'))

ax.set_xlabel('z of probe (Å)', fontsize=12)
ax.set_ylabel('Coordination number (CN)', fontsize=12)
ax.set_title('Panel 2: CN vs z during noble-gas drop into void', fontsize=12)
ax.invert_xaxis()   # left = high z (above surface), right = low z (deep in void)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('panel2_cn_vs_z.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: panel2_cn_vs_z.png')

## Panel 3 — Unique sites inside the void

**Method:** Collect all unique probe positions from the drop trajectory (PBC-aware, within `uniq_tol`).  
Each unique position is a candidate adsorption site inside the void.

**Trajectory:** one frame per unique site.  
**Figure:** side-view (x vs z), probe positions coloured by CN.

In [ ]:
# Panel 3 — unique sites trajectory
uniq_frames = []
uniq_meta   = []

for f, m in zip(frames_drop, meta_drop):
    pos_f = f.positions[i_tag]
    keep  = True
    for u in uniq_frames:
        pos_u = u.positions[i_tag]
        _, d  = get_distances([pos_f], [pos_u], cell=f.cell, pbc=f.pbc)
        if float(d) < uniq_tol:
            keep = False
            break
    if keep:
        uniq_frames.append(f.copy())
        uniq_meta.append(m)

print(f'Unique probe positions: {len(uniq_frames)}')
for m in uniq_meta:
    print(f"  step {m['step']:3d}  z = {m['z']:.3f} Å  CN = {m['cn']}")

write('panel3_void_sites.traj', uniq_frames)
print('\nWrote: panel3_void_sites.traj')

In [ ]:
# Panel 3 — side-view matplotlib figure
slab_x = slab.positions[:, 0]
slab_z = slab.positions[:, 2]

cn_all = np.array([m['cn'] for m in uniq_meta], dtype=float)
z_all  = np.array([uniq_frames[k].positions[i_tag, 2] for k in range(len(uniq_frames))])
x_all  = np.array([uniq_frames[k].positions[i_tag, 0] for k in range(len(uniq_frames))])

cmap = plt.cm.plasma
norm = Normalize(vmin=cn_all.min() - 0.5, vmax=cn_all.max() + 0.5)

fig, ax = plt.subplots(figsize=(7, 4))

ax.scatter(slab_x, slab_z, c='lightsteelblue', s=130, edgecolors='gray',
           linewidths=0.5, zorder=2, label='slab atoms')

sc = ax.scatter(x_all, z_all, c=cn_all, cmap=cmap, norm=norm,
                s=200, edgecolors='k', linewidths=0.8,
                marker='D', zorder=4, label='probe positions')
plt.colorbar(sc, ax=ax, label='Coordination number (CN)')

# highlight the max-CN site
best_idx = int(np.argmax(cn_all))
ax.scatter(x_all[best_idx], z_all[best_idx],
           c='gold', s=350, marker='*', edgecolors='k', linewidths=1.0,
           zorder=5, label=f'max CN = {int(cn_all[best_idx])}')

# label each site with its CN
for k in range(len(uniq_frames)):
    ax.annotate(f"CN={int(cn_all[k])}",
                xy=(x_all[k], z_all[k]),
                xytext=(x_all[k] + 0.15, z_all[k] + 0.15),
                fontsize=7, color='k')

ax.set_xlabel('x (Å)', fontsize=12)
ax.set_ylabel('z (Å)', fontsize=12)
ax.set_title('Panel 3: Side view — unique probe sites inside void', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig('panel3_void_sites.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: panel3_void_sites.png')